<a href="https://colab.research.google.com/github/RobJavVar/DataSciencePsychNeuro/blob/master/ExerciseSubmissions/18_principal-component-methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 18: Principal component methods

This homework assignment is designed to give you an intuition on principal component approaches to understanding high dimensional statistical relationships.

Like earlier homework, you will need to download the **unrestricted_trimmed_1_7_2020_10_50_44.csv** file from the *Homework/hcp_data* folder in the class GitHub repository.

This data is a portion of the [Human Connectome Project database](http://www.humanconnectomeproject.org/). It provides measures of cognitive tasks and brain morphology measuresments from 1206 participants. The full description of each variable is provided in the **HCP_S1200_DataDictionary_April_20_2018.csv** file in the *Homework/hcp_data* folder in the class GitHub repository.

---
## 1. Loading data (1 point)

We are going to look for low dimensional relationships between brain volume measures and working memory capacity.

First, we will need to load the `pls`, `tidyverse`, and `ggplot2` libraries for this assignment.


In [ ]:
library(pls)
library(tidyverse)
library(ggplot2)
library(dplyr)


Use `read.csv` function to load data from the **unrestricted_trimmed_1_7_2020_10_50_44.csv** file in the **hcp_data** folder.

(a) Using the tidyverse tools, make a new dataframe `d1` that only includes the Flanker Task performance (`Flanker_Unadj`) and *all* freesurfer **volume** measures for the right and left hemispheres together. Remove both "na's" and any columns that consist of only zeros.

Hint: Look up using the `ends_with` function to only select variables that end with "_Vol"

Use the `head` function to look at the first few rows of each data frame.

In [ ]:
# setwd("~/Documents/GitHub/DataSciencePsychNeuro/Exercise datasets/hcp_data")
# lex <- read.csv("unrestricted_trimmed_1_7_2020_10_50_44.csv")
# library(tidyverse)
d1 <- lex %>%
    select(Flanker_Unadj, ends_with("_Vol")) %>%
    drop_na() %>%
    select(where(~ !is.numeric(.x) || any(.x != 0)))
head(d1)
dim(d1)


---
## 2. Correlational structure (4 points)

(a) Take a look at the correlation between all of the freesurfer volume measures ("FS_") using the `cor` function. Create a new variable called `fs_cor` that is the correlation matrix for only the freesurfer volumes

In [ ]:
fs_data <- d1 %>%
  select(starts_with("FS_")) 
fs_cor <- fs_data %>%
  cor(use = "complete.obs")

print(fs_cor)


(b) Load (and install locally if needed) the `reshape2` library in order to use the `melt` function on the new `fs_cor` object. Use `head` to show the new, melted `fs_cor` object.

In [ ]:
library(reshape2)
fs_cor_melted <- melt(fs_cor)
head(fs_cor_melted)


Plot the correlation as a heatmap using `ggplot2`.

Hint: use the `scale_fill_gradient2` function to scale the colors between red and blue, capping the values at -1 and +1.

In [ ]:
ggplot(data = fs_cor_melted, aes(x = Var1, y = Var2, fill = value)) +
  geom_tile(color = "white") +
  scale_fill_gradient2(low = "blue", mid = "white", high = "red", 
                       midpoint = 0, limit = c(-1, 1), 
                       name = "Correlation") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, vjust = 1, hjust = 1),
        axis.title.x = element_blank(), # Optional: removes redundant axis titles
        axis.title.y = element_blank()) +
  coord_fixed() +
  labs(title = "Correlation Heatmap of FreeSurfer Volumes")


What patterns do you see in the correlations?

> Almost all of the correlations are positive, and the strongest correlations are between the WM and GM volume measures.
> *


---
## 3. Principal component analysis (3 points)

Let's see how many principal components explain at *at least* 95% of the data.

(a) Create a new object called `fs_d.pca` using the `princomp` function (do not forget to scale the data).


In [ ]:
fs_d.pca <- princomp(fs_data, cor = TRUE)
summary(fs_d.pca)


(b) Calculate the cumulative variance explained (not unique variance explained as in the tutorial) across the principal components and plot the results using `ggplot`.

In [ ]:
pca_var <- fs_d.pca$sdev^2
pca_prop_var <- pca_var / sum(pca_var)
pca_cum_var <- cumsum(pca_prop_var)

pca_df <- data.frame(
  Component = 1:length(pca_cum_var),
  Cumulative_Variance = pca_cum_var
)

ggplot(data = pca_df, aes(x = Component, y = Cumulative_Variance)) +
  geom_line(color = "blue", size = 1) +
  geom_point(color = "red", size = 3) +
  scale_x_continuous(breaks = 1:length(pca_cum_var)) +
  scale_y_continuous(limits = c(0, 1), breaks = seq(0, 1, by = 0.1)) +
  theme_minimal() +
  labs(
    title = "Cumulative Variance Explained by Principal Components",
    x = "Principal Component",
    y = "Cumulative Variance Explained"
  )


(c) Determine exactly how many principal components explain at least 95% of the variance.

Hint: Look up the `which` function.

In [ ]:
components_95 <- which(pca_cum_var >= 0.95)

min_95 <- components_95[1]

print(min_95)


What does this tell you about the underlying dimensionality of the brain volume measures?

> The dimensionality is much smaller than the original number of variables. This is because there is a lot of covariance between the original variables, such as the WM and GM measures we saw light up in the heatmap. Using PCA helps simplify and avoid multicollinearity.
> 


---
## 4. Associating with Flanker task peformance (4 points)

Now apply PCR to the `d1` object you created at the beginning (which includes the Flanker task scores) to find how freesurfer volumes predict Flanker task performance. Set the random seed to "2". Use cross-validation as the validation type and don't forget to scale your data. Show the `summary` of the model fit.

Hint: If you receive an error applying the "scale=TRUE" flag, then you likely still have columns of all zeros somewhere in your data table.

In [ ]:
set.seed(2)

pcr_data <- d1 %>%
  select(Flanker_Unadj, starts_with("FS_"))

pcr_fit <- pcr(Flanker_Unadj ~ ., 
               data = pcr_data, 
               scale = TRUE, 
               validation = "CV")

summary(pcr_fit)


(b) Use the `validationplot` function to evaluate the bias-variance tradeoff using the cross-validated mean squared-error for each component.

In [ ]:
validationplot(pcr_fit, 
               val.type = "RMSEP", 
               legendpos = "topright",
               main = "Cross-Validation Plot: Bias-Variance Tradeoff")


(c) Extract the MSEP values of the cross validated fit ("CV" not "adj_CV") from the model object using the `MSEP` funtion. Create a new array of these values. Use the `drop` function to remove the singleton dimension (the original array is 2x1x53 and we want a 2x53 object). Find the minimum value of the first row (the "CV").

In [ ]:
msep_obj <- MSEP(pcr_fit)

msep_array <- drop(msep_obj$val)

cv_values <- msep_array[1, ] 

min_msep <- min(cv_values)

print(min_msep)


What what does this plot tell you about how many components best explain variance in Flanker task performance?

> 7 components is the best number to avoid overfitting while explaining variance in the flanker task performance. parsimony!
> 

---
## 5. Reflection (2 points)

Compare the number of components that explain variance in X alone (the brain volumes) to the number of components that explain performance in the Flanker task. What does the difference in these two numbers tell you about how variation in brain volumes relates to task performance?

> While we needed a lot of components to model the brains structure, much less complexity is necessary when we're focusing on actual executive functions. It's easy to simplify these functions into only 7 components rather than 27. We are probably overfitting to hyper specific biological details with PCA, because once it is based on an actual task the brain appears much simpler.

>

**DUE:** 5pm EST, April 9, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> I asked gemini for help with the code